![Header Image](../assets/header_image.png "Header Image")

# Devoir Optionnel : Introduction à ROS 2

Bienvenue dans ce tutoriel d'introduction à **ROS 2** (Robot Operating System 2) !
Cet exercice est facultatif. Nous le recommandons aux étudiants qui souhaitent découvrir la version moderne de ROS, en particulier ceux qui :
- ont déjà une expérience de base avec **ROS 1**
- ou qui débutent directement avec **ROS 2**

Dans ce devoir, vous allez

- **comprendre les différences fondamentales entre ROS 1 et ROS 2**
- **apprendre à initialiser un nœud ROS 2 dans un Jupyter Notebook avec `rclpy`**
- **pratiquer la communication Publisher-Subscriber avec différents types de messages**
- **utiliser le threading pour exécuter un nœud ROS 2 de manière asynchrone dans Jupyter**

# ROS 2 vs ROS 1 : Différences Clés

ROS 2 est la nouvelle génération du Robot Operating System. Il a été conçu pour corriger les limites de ROS 1 et répondre aux besoins des systèmes robotiques modernes.

| Aspect | ROS 1 | ROS 2 |
|--------|--------|--------|
| **Middleware** | Architecture centralisée (roscore) | DDS (Data Distribution Service) décentralisé |
| **Nœud maître** | `roscore` obligatoire | **Aucun master requis** |
| **Bibliothèque Python** | `rospy` | `rclpy` |
| **Temps réel** | Non supporté nativement | Supporté |
| **Sécurité** | Absente | Intégrée (DDS-Security) |
| **Systèmes d'exploitation** | Linux uniquement | Linux, Windows, macOS |
| **Communication** | TCP/UDP via roscore | DDS (peer-to-peer) |
| **Actions** | Bibliothèque séparée | Intégrée nativement |
| **Distribution (ce conteneur)** | **Noetic** (Ubuntu 20.04) | **Foxy** (Ubuntu 20.04) |

La différence la plus importante pour ce tutoriel : **il n'est plus nécessaire de lancer `roscore` avant de démarrer vos nœuds.**

# Configurer l'environnement ROS 2

La commande suivante configure l'environnement pour la distribution **Foxy Fitzroy** de ROS 2,
installée dans ce conteneur Docker (Ubuntu 20.04 Focal).
Elle initialise les variables d'environnement nécessaires (`AMENT_PREFIX_PATH`, `ROS_DISTRO`, etc.).

Contrairement à ROS 1 qui utilisait `catkin`, ROS 2 utilise le système de build **colcon** et le gestionnaire de packages **ament**.

In [1]:
!source /opt/ros/foxy/setup.bash

# Ajouter le chemin de la bibliothèque Python de ROS 2

La bibliothèque cliente Python de ROS 2 s'appelle **`rclpy`** (ROS Client Library for Python).

Dans ce conteneur Docker, ROS 2 Foxy est compilé pour **Python 3.8** (Python système Ubuntu 20.04).
Ce notebook utilise le kernel **"Python 3.8 (ROS 2 Foxy)"** — assurez-vous de l'avoir sélectionné en haut à droite de JupyterLab.

> Si le kernel affiché est "Python 3" (conda 3.9), les imports `rclpy` échoueront avec une erreur d'extension C.
> Changez-le via **Kernel >> Change Kernel >> Python 3.8 (ROS 2 Foxy)**.

In [1]:
import sys
sys.path.insert(0, '/opt/ros/foxy/lib/python3.8/site-packages/')

# Vérifier que le bon Python est utilisé (doit afficher 3.8.x)
import platform
print("Python utilisé :", platform.python_version())
assert platform.python_version_tuple()[1] == '8', \
    "ERREUR : mauvais kernel ! Sélectionnez 'Python 3.8 (ROS 2 Foxy)' dans Kernel >> Change Kernel."
print("Kernel OK — Python 3.8 confirmé.")

Python utilisé : 3.8.10
Kernel OK — Python 3.8 confirmé.


# Importation des bibliothèques ROS 2 et des types de messages

En ROS 2, la bibliothèque Python principale est **`rclpy`** (à la place de `rospy` en ROS 1).
Les types de messages restent compatibles : `std_msgs`, `geometry_msgs`, `sensor_msgs`.

Nous importons également **`threading`** qui nous permettra d'exécuter le nœud ROS 2 en arrière-plan,
car `rclpy.spin()` est une fonction bloquante.

In [2]:
import rclpy
from rclpy.node import Node

from std_msgs.msg import String
from geometry_msgs.msg import Pose, Twist
from sensor_msgs.msg import Image

import threading
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Bibliothèques importées avec succès !")

Matplotlib created a temporary cache directory at /tmp/matplotlib-sv7ophee because the default path (/home/jovyan/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Bibliothèques importées avec succès !


# Introduction à ROS 2 (Robot Operating System 2)

ROS 2 (https://docs.ros.org/en/foxy/) est la nouvelle génération du Robot Operating System.
Il conserve les principes de ROS 1 (modularité, communication par messages) tout en apportant des améliorations majeures :

- **Architecture décentralisée** basée sur DDS (Data Distribution Service)
- **Support multi-plateforme** (Linux, Windows, macOS)
- **Temps réel** natif
- **Sécurité intégrée** avec DDS-Security
- **Cycle de vie des nœuds** (Lifecycle Nodes)
- **Actions** intégrées nativement

# Concepts Fondamentaux de ROS 2

## Niveau du système de fichiers

### Paquets (Packages) :
Les paquets restent l'unité de base d'organisation du code en ROS 2. Ils utilisent maintenant le système **ament** (à la place de catkin) et contiennent un fichier `package.xml` et un `CMakeLists.txt` ou `setup.py`.

## Niveau du graphe de communication

ROS 2 remplace l'architecture centralisée de ROS 1 par un middleware **DDS** (Data Distribution Service) décentralisé.

### Nœuds (Nodes) :
Les nœuds sont des processus qui effectuent des calculs. En ROS 2, chaque nœud communique directement avec les autres via DDS, **sans passer par un master central**.

### Topics :
Les topics fonctionnent de la même manière qu'en ROS 1 : un éditeur (publisher) publie des messages sur un topic, et un ou plusieurs abonnés (subscribers) les reçoivent.

### Services :
Les services permettent une communication requête/réponse entre nœuds. En ROS 2, ils sont plus robustes qu'en ROS 1.

### Actions :
Les actions sont une nouveauté importante de ROS 2. Elles permettent d'exécuter des tâches longues avec feedback en temps réel et possibilité d'annulation.

### Paramètres :
En ROS 2, les paramètres sont associés directement à chaque nœud (pas de serveur de paramètres global comme en ROS 1).

### Bags :
ROS 2 utilise la commande `ros2 bag` (à la place de `rosbag`) pour enregistrer et rejouer des données. Le format de fichier est SQLite3 ou MCAP.

# Pas besoin de lancer un nœud maître !

C'est l'une des différences les plus importantes entre ROS 1 et ROS 2.

En **ROS 1**, il fallait obligatoirement lancer `roscore` dans un terminal avant de pouvoir créer des nœuds :
```bash
# ROS 1 uniquement — NON requis en ROS 2
source /opt/ros/noetic/setup.sh && roscore
```

En **ROS 2**, grâce au middleware DDS, les nœuds se découvrent automatiquement. Il suffit d'initialiser `rclpy` dans votre code Python.

> **En résumé :** En ROS 2, vous pouvez commencer à coder immédiatement, sans ouvrir de terminal supplémentaire pour lancer un master.

# Initialiser rclpy et créer un nœud ROS 2

En ROS 2, l'initialisation se fait en deux étapes :
1. **`rclpy.init()`** : initialise le middleware DDS
2. **`rclpy.create_node('nom_du_noeud')`** : crée un nœud ROS 2

Il est aussi possible de créer une classe héritant de `Node`, ce qui est la pratique recommandée pour les projets plus complexes.

In [3]:
# Initialiser rclpy (obligatoire avant toute opération ROS 2)
rclpy.init()

# Créer un nœud nommé 'jupyter_node'
node = rclpy.create_node('jupyter_node')

print(f"Nœud '{node.get_name()}' créé avec succès !")
print("Aucun roscore nécessaire — ROS 2 utilise DDS.")

Nœud 'jupyter_node' créé avec succès !
Aucun roscore nécessaire — ROS 2 utilise DDS.


# Threading pour Jupyter Notebook

Dans un Jupyter Notebook, le code s'exécute de manière séquentielle.
Or, `rclpy.spin(node)` est une **fonction bloquante** qui attend en permanence des messages entrants.

Pour pouvoir continuer à exécuter d'autres cellules tout en recevant des messages, nous lançons le spin ROS 2 dans un **thread séparé** en arrière-plan.

- `daemon=True` : le thread s'arrêtera automatiquement à la fermeture du notebook
- Ce thread gère la réception et le dispatch des messages vers les callbacks

In [4]:
# Lancer le spin ROS 2 dans un thread d'arrière-plan
spin_thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
spin_thread.start()

print("Thread ROS 2 démarré en arrière-plan.")
print("Le nœud est maintenant actif et prêt à envoyer/recevoir des messages.")

Thread ROS 2 démarré en arrière-plan.
Le nœud est maintenant actif et prêt à envoyer/recevoir des messages.


# Communication Publisher-Subscriber

## Publisher :
Un publisher est associé à un nœud et permet de **publier** des messages ROS 2 sur un topic.

En ROS 2 avec `rclpy`, un publisher se crée avec :
```python
node.create_publisher(TypeMessage, 'nom_topic', profondeur_file_attente)
```

## Subscriber :
Un subscriber est associé à un nœud et permet de **recevoir** des messages publiés sur un topic.

En ROS 2 avec `rclpy`, un subscriber se crée avec :
```python
node.create_subscription(TypeMessage, 'nom_topic', fonction_callback, profondeur_file_attente)
```

La `profondeur_file_attente` (QoS — Quality of Service) définit combien de messages peuvent être mis en file d'attente. Une valeur de `10` est courante.

L'image suivante visualise la communication entre Publisher et Subscriber ainsi que les nœuds et les topics en ROS 2 (pas de master — DDS gère la découverte automatiquement) :

```
┌─────────────────────────────────────────────────────────────┐
│                    Réseau DDS (ROS 2)                        │
│                                                             │
│   ┌──────────────┐    /pose_stream    ┌──────────────────┐  │
│   │   Publisher  │ ────────────────► │   Subscriber     │  │
│   │ (jupyter_    │                   │  (jupyter_node)  │  │
│   │    node)     │                   │                  │  │
│   └──────────────┘                   └──────────────────┘  │
│                                                             │
│              Découverte automatique via DDS                 │
└─────────────────────────────────────────────────────────────┘
```

# Publier sur un topic ROS 2

Nous allons publier des messages `Pose` sur le topic `/pose_stream`.
Le message [`Pose`](https://docs.ros2.org/foxy/api/geometry_msgs/msg/Pose.html) est défini comme :

```
Point position
    float64 x
    float64 y
    float64 z
Quaternion orientation
    float64 x
    float64 y
    float64 z
    float64 w
```

Contrairement à ROS 1 avec JupyROS qui générait des widgets automatiquement, en ROS 2 nous créons et publions les messages directement en Python.

In [5]:
# Créer un publisher pour des messages Pose sur le topic /pose_stream
pub_pose = node.create_publisher(Pose, '/pose_stream', 10)

# Construire le message
msg_pose = Pose()
msg_pose.position.x = 1.0
msg_pose.position.y = 2.5
msg_pose.position.z = 0.0
msg_pose.orientation.x = 0.0
msg_pose.orientation.y = 0.0
msg_pose.orientation.z = 0.0
msg_pose.orientation.w = 1.0  # Quaternion identité (pas de rotation)

# Publier le message
pub_pose.publish(msg_pose)

print("Message Pose publié sur /pose_stream :")
print(f"  Position    : x={msg_pose.position.x}, y={msg_pose.position.y}, z={msg_pose.position.z}")
print(f"  Orientation : w={msg_pose.orientation.w} (identité — pas de rotation)")

Message Pose publié sur /pose_stream :
  Position    : x=1.0, y=2.5, z=0.0
  Orientation : w=1.0 (identité — pas de rotation)


# S'abonner à un topic ROS 2

Nous allons maintenant créer un subscriber pour recevoir les messages `Pose` publiés sur `/pose_stream`.

La **fonction de callback** est appelée automatiquement par le thread ROS 2 en arrière-plan chaque fois qu'un nouveau message arrive.

Exécutez la cellule ci-dessous, puis **ré-exécutez la cellule de publication** ci-dessus pour voir les messages arriver !

In [6]:
messages_pose = []

def callback_pose(msg):
    messages_pose.append(msg)
    print(f"[/pose_stream] Reçu — position: ({msg.position.x:.2f}, {msg.position.y:.2f}, {msg.position.z:.2f})")

sub_pose = node.create_subscription(Pose, '/pose_stream', callback_pose, 10)

print("Subscriber actif sur /pose_stream — en attente de messages...")
print("Ré-exécutez la cellule de publication pour voir les messages arriver.")

Subscriber actif sur /pose_stream — en attente de messages...
Ré-exécutez la cellule de publication pour voir les messages arriver.


Lors de la création du subscriber, `rclpy` enregistre le callback dans le nœud. Le thread d'arrière-plan appelle automatiquement ce callback à chaque réception de message.

C'est l'équivalent ROS 2 de :
```python
# ROS 1 (JupyROS)
jr.subscribe('/pose_stream', Pose, lambda msg: print(msg))
```

En ROS 2, nous avons un contrôle plus direct et explicite sur le callback, sans dépendre d'une bibliothèque tierce comme JupyROS.

# Utiliser différents types de messages

Nous pouvons utiliser n'importe quel type de message ROS 2 standard.
Essayons avec :
- [`Twist`](https://docs.ros2.org/foxy/api/geometry_msgs/msg/Twist.html) — commandes de vitesse (très utilisé pour les robots mobiles)
- [`String`](https://docs.ros2.org/foxy/api/std_msgs/msg/String.html) — messages texte simples

In [7]:
# Publisher Twist sur /twist_stream
pub_twist = node.create_publisher(Twist, '/twist_stream', 10)

msg_twist = Twist()
msg_twist.linear.x  = 0.5   # Vitesse linéaire en m/s
msg_twist.angular.z = 0.3   # Vitesse angulaire en rad/s

pub_twist.publish(msg_twist)
print(f"Twist publié — linéaire: {msg_twist.linear.x} m/s, angulaire: {msg_twist.angular.z} rad/s")

Twist publié — linéaire: 0.5 m/s, angulaire: 0.3 rad/s


In [8]:
# Publisher String sur /string_stream
pub_string = node.create_publisher(String, '/string_stream', 10)

msg_string = String()
msg_string.data = "Bonjour depuis ROS 2 Foxy et Jupyter Notebook !"

pub_string.publish(msg_string)
print(f"String publié : '{msg_string.data}'")

String publié : 'Bonjour depuis ROS 2 Foxy et Jupyter Notebook !'


In [9]:
# Subscribers pour Twist et String
messages_twist  = []
messages_string = []

def callback_twist(msg):
    messages_twist.append(msg)
    print(f"[/twist_stream]  Reçu — linéaire: {msg.linear.x:.2f} m/s, angulaire: {msg.angular.z:.2f} rad/s")

def callback_string(msg):
    messages_string.append(msg.data)
    print(f"[/string_stream] Reçu — '{msg.data}'")

sub_twist  = node.create_subscription(Twist,  '/twist_stream',  callback_twist,  10)
sub_string = node.create_subscription(String, '/string_stream', callback_string, 10)

print("Subscribers actifs sur /twist_stream et /string_stream.")
print("Ré-exécutez les cellules de publication pour voir les messages.")

Subscribers actifs sur /twist_stream et /string_stream.
Ré-exécutez les cellules de publication pour voir les messages.


# Publier et visualiser des images

Nous pouvons également publier des images en utilisant le type de message [`sensor_msgs/Image`](https://docs.ros2.org/foxy/api/sensor_msgs/msg/Image.html).

La structure du message `Image` est :
```
std_msgs/Header header
uint32 height
uint32 width
string encoding
uint8 is_bigendian
uint32 step
uint8[] data
```

La cellule suivante charge l'image d'en-tête du cours, remplit un message `Image` ROS 2 et le publie sur `/image_stream`.

> Dans ce conteneur Docker, le dépôt est monté dans `/home/jovyan/tp-va/`.

In [10]:
# Charger une image et la publier comme message ROS 2 Image
image_path = '/home/jovyan/tp-va/assets/header_image.png'
cv_image = cv2.imread(image_path)

if cv_image is not None:
    pub_image = node.create_publisher(Image, '/image_stream', 10)

    msg_image = Image()
    msg_image.height   = cv_image.shape[0]
    msg_image.width    = cv_image.shape[1]
    msg_image.encoding = 'bgr8'
    msg_image.step     = cv_image.shape[1] * 3
    msg_image.data     = cv_image.tobytes()

    pub_image.publish(msg_image)
    print(f"Image publiée sur /image_stream ({msg_image.width}x{msg_image.height} px)")
else:
    print(f"Image non trouvée à : {image_path}")

Image publiée sur /image_stream (1200x242 px)


## Recevoir et afficher les images

La fonction `afficher_image_ros2()` prend le message `Image` ROS 2 en entrée,
reconstruit un tableau NumPy à partir des données brutes, et affiche l'image avec `matplotlib`.

In [11]:
def afficher_image_ros2(img_msg):
    np_image = np.frombuffer(img_msg.data, dtype=np.uint8)
    np_image = np_image.reshape((img_msg.height, img_msg.width, 3))
    np_image = np_image[..., ::-1]  # BGR (OpenCV) → RGB (matplotlib)
    plt.figure(figsize=(8, 4))
    plt.imshow(np_image)
    plt.axis('off')
    plt.title('Image reçue depuis /image_stream')
    plt.show()
    plt.close()

sub_image = node.create_subscription(Image, '/image_stream', afficher_image_ros2, 10)

print("Subscriber actif sur /image_stream.")
print("Ré-exécutez la cellule de publication d'image pour voir l'affichage.")

Subscriber actif sur /image_stream.
Ré-exécutez la cellule de publication d'image pour voir l'affichage.


# Vérifier les topics actifs depuis le terminal

Vous pouvez vérifier les topics ROS 2 actifs depuis un terminal JupyterLab (**Fichier >> Nouveau >> Terminal**) :

```bash
source /opt/ros/foxy/setup.bash
ros2 topic list
```

Vous devriez voir :
```
/image_stream
/parameter_events
/pose_stream
/rosout
/string_stream
/twist_stream
```

Pour voir les messages en temps réel :
```bash
ros2 topic echo /string_stream
ros2 topic echo /pose_stream
```

Pour obtenir des informations sur un topic :
```bash
ros2 topic info /pose_stream
```

> **Rappel :** Contrairement à ROS 1, **aucun `roscore` n'est nécessaire** pour que ces commandes fonctionnent.

# Arrêter le nœud et libérer les ressources

À la fin de votre session, il est important de :
1. **Détruire le nœud** avec `node.destroy_node()`
2. **Arrêter rclpy** avec `rclpy.shutdown()`

Cela libère proprement les ressources DDS. Le thread d'arrière-plan (daemon) s'arrêtera automatiquement.

In [12]:
node.destroy_node()
rclpy.shutdown()

print("Nœud ROS 2 arrêté proprement.")
print("rclpy shutdown — ressources DDS libérées.")

Nœud ROS 2 arrêté proprement.
rclpy shutdown — ressources DDS libérées.


# Résumé

- Vous avez appris les **différences fondamentales entre ROS 1 et ROS 2** : architecture DDS, absence de master, `rclpy` vs `rospy`.
- Vous avez appris à **initialiser `rclpy`** et à créer un nœud ROS 2 directement dans un Jupyter Notebook.
- Vous avez appris à utiliser le **threading** pour exécuter `rclpy.spin()` en arrière-plan, permettant la réception asynchrone de messages.
- Vous avez appris à créer des **publishers** avec `node.create_publisher()` pour envoyer des messages de types `Pose`, `Twist`, `String` et `Image`.
- Vous avez appris à créer des **subscribers** avec `node.create_subscription()` et à définir des fonctions de callback.
- Vous avez appris à **arrêter proprement** un nœud ROS 2 avec `node.destroy_node()` et `rclpy.shutdown()`.